# 03 — Exploratory Data Analysis
**Distributions · Skill Frequency · Match Score Analysis · Correlations**

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='viridis')
df = pd.read_csv('../data/processed/jobs_cleaned.csv')
print(f"Shape: {df.shape}")
df.head(3)

## 1. Recommendation Rate (Key KPI)

In [ ]:
rec_rate = df['recommended'].mean()
print(f"Overall Recommendation Rate: {rec_rate:.2%}")
df['recommended'].value_counts().plot(kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='black')
plt.title('Recommendation Outcome Distribution')
plt.xticks([0,1], ['Not Recommended','Recommended'], rotation=0)
plt.ylabel('Count')
plt.tight_layout(); plt.savefig('../tableau/screenshots/01_recommendation_dist.png', dpi=150); plt.show()

## 2. Match Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['match_score'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Match Score Histogram'); axes[0].set_xlabel('Match Score')
df.boxplot(column='match_score', by='recommended', ax=axes[1], patch_artist=True)
axes[1].set_title('Match Score by Recommendation'); axes[1].set_xlabel('Recommended (0/1)')
plt.suptitle('')
plt.tight_layout(); plt.savefig('../tableau/screenshots/02_match_score_dist.png', dpi=150); plt.show()
print(df.groupby('recommended')['match_score'].describe().round(2))

## 3. Match Score Band Analysis

In [ ]:
band_rec = df.groupby('match_score_band')['recommended'].mean().reset_index()
band_rec.columns = ['Band','Rec Rate']
print(band_rec)
band_rec.plot(x='Band', y='Rec Rate', kind='bar', color='teal', edgecolor='black', legend=False)
plt.title('Recommendation Rate by Match Score Band')
plt.ylabel('Recommendation Rate')
plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig('../tableau/screenshots/03_band_rec_rate.png', dpi=150); plt.show()

## 4. Skill Count Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title in zip(axes,
    ['user_skill_count','job_req_count','skill_overlap_count'],
    ['User Skill Count','Job Req Count','Skill Overlap']):
    df[col].hist(bins=30, ax=ax, color='mediumpurple', edgecolor='white')
    ax.set_title(title)
plt.tight_layout(); plt.savefig('../tableau/screenshots/04_skill_counts.png', dpi=150); plt.show()
print(df[['user_skill_count','job_req_count','skill_overlap_count']].describe().round(2))

## 5. Top Skills — Candidates vs Jobs

In [ ]:
user_skills = df['user_skills'].str.split(',').explode().str.strip().loc[lambda s: s!='']
job_skills  = df['job_requirements'].str.split(',').explode().str.strip().loc[lambda s: s!='']
top_user = user_skills.value_counts().head(15)
top_job  = job_skills.value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_user.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Candidate Skills'); axes[0].invert_yaxis()
top_job.plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Top 15 Job Requirements'); axes[1].invert_yaxis()
plt.tight_layout(); plt.savefig('../tableau/screenshots/05_top_skills.png', dpi=150); plt.show()

skill_freq = pd.concat([top_user.rename('user_freq'), top_job.rename('job_freq')], axis=1).fillna(0)
skill_freq.to_csv('../data/processed/skill_frequency.csv')
print("Saved skill_frequency.csv")

## 6. Correlation Matrix

In [ ]:
num_cols = ['match_score','user_skill_count','job_req_count','skill_overlap_count','overlap_ratio','skill_gap','recommended']
corr = df[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout(); plt.savefig('../tableau/screenshots/06_correlation.png', dpi=150); plt.show()

## 7. EDA Insights
- Candidates with **High** match score band are recommended at a significantly higher rate
- `match_score` and `skill_overlap_count` are the strongest predictors of recommendation
- There is a visible skill gap: job requirements often exceed candidate skill counts
- Top demanded skills differ from top candidate skills — representing a market opportunity